# Tightloop

Tightloop was a way of pre-calculating all information required to do a given contraction. No tensorkit overhead, no digging and calculating, it'd all be specialized given the spaces. The usecase were "tight loops", evaluating the same contraction over and over again, like in an eigenvalue problem. It used to give a noticeable speedup but through voodoo and witchcraft, native TensorKit is now comfortably beating TightLoop on everything;

I'm keeping the notebook around for posterity.

In [1]:
using Revise, TensorKit, TensorOperations, BenchmarkTools, Strided, TupleTools, MPSKitExperimental


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
import LinearAlgebra
using Base.Threads
LinearAlgebra.BLAS.set_num_threads(1)
Strided.set_num_threads(1)
nthreads()

4

## Practical benchmarks

In [3]:
@tightloop_tensor ac_eff y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]
@tightloop_tensor ac_eff_malloc allocator=malloc y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]

In [4]:
virtspace = Rep[SU₂](i => 20 for i in 0:10);
ospace = Rep[SU₂](0 => 5,1 => 2);
pspace = Rep[SU₂](1 => 1);

t_le = rand(ComplexF64,virtspace*ospace',virtspace);
t_re = rand(ComplexF64,virtspace*ospace,virtspace);
t_ac = rand(ComplexF64,virtspace*pspace,virtspace);
t_o = rand(ComplexF64,ospace*pspace,pspace*ospace);

factory = ac_eff(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(t_ac),space(t_ac)));
factory_malloc = ac_eff_malloc(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(t_ac),space(t_ac)));

In [5]:
@benchmark $factory_malloc(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 1976 samples with 1 evaluation per sample.
 Range (min … max):  2.088 ms …  10.925 ms  ┊ GC (min … max): 0.00% … 75.92%
 Time  (median):     2.216 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.530 ms ± 826.861 μs  ┊ GC (mean ± σ):  2.89% ±  7.69%

  ██▆▄▃▂▁▁▁▃▂▂▂                                               ▁
  ███████████████▆▇▇▆▇▆▅▇▆███▇▇▆▆▆▅▄▁▁▁▄▁▄▄▁▁▁▁▁▄▁▁▁▁▁▁▁▄▄▄▆▆ █
  2.09 ms      Histogram: log(frequency) by time      6.88 ms <

 Memory estimate: 472.77 KiB, allocs estimate: 4713.

In [6]:
@benchmark $factory(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 896 samples with 1 evaluation per sample.
 Range (min … max):  2.867 ms … 111.403 ms  ┊ GC (min … max):  0.00% … 96.61%
 Time  (median):     3.122 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):   5.578 ms ±  11.834 ms  ┊ GC (mean ± σ):  35.42% ± 19.37%

  █▅▂▁                                                         
  ████▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄ ▇
  2.87 ms      Histogram: log(frequency) by time       106 ms <

 Memory estimate: 11.78 MiB, allocs estimate: 4717.

In [7]:
function slowcontract(;le=nothing,re=nothing,x=nothing,O=nothing)
    @tensor allocator=malloc y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]
end
@benchmark slowcontract(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 1123 samples with 1 evaluation per sample.
 Range (min … max):  4.075 ms …  12.000 ms  ┊ GC (min … max): 0.00% … 55.08%
 Time  (median):     4.305 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.449 ms ± 711.930 μs  ┊ GC (mean ± σ):  1.41% ±  5.62%

  ▂█▅                                                          
  ████▅▄▃▃▃▂▂▂▂▂▂▂▂▁▁▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂ ▂
  4.08 ms         Histogram: frequency by time        10.2 ms <

 Memory estimate: 472.21 KiB, allocs estimate: 4716.

In [8]:
norm(slowcontract(le = t_le, re=t_re, x = t_ac, O = t_o)- factory(le = t_le, re=t_re, x = t_ac, O = t_o))

0.0

In [9]:
virtspace = Rep[SU₂×U₁×ℤ₂]((i,j,b) => 10 for i in 0:5,j in -3:3,b in 0:1);
ospace = Rep[SU₂×U₁×ℤ₂]((i,j,b) => 2 for i in 0:1,j in -1:1,b in 0:1);
pspace = Rep[SU₂×U₁×ℤ₂]((i,j,b) => 1 for i in 1:1,j in 0:0,b in 0:0);

t_le = rand(ComplexF64,virtspace*ospace',virtspace);
t_re = rand(ComplexF64,virtspace*ospace,virtspace);
t_ac = rand(ComplexF64,virtspace*pspace,virtspace);
t_o = rand(ComplexF64,ospace*pspace,pspace*ospace);

factory = ac_eff_malloc(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(t_ac),space(t_ac)));

In [10]:
@benchmark $factory(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 129 samples with 1 evaluation per sample.
 Range (min … max):  36.797 ms … 46.248 ms  ┊ GC (min … max): 0.00% … 15.96%
 Time  (median):     37.341 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   38.816 ms ±  2.975 ms  ┊ GC (mean ± σ):  3.28% ±  6.25%

  ▂█▃▅                                                         
  ████▄▄▃▅▃▃▃▃▄▄▃▄▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▇▄▃▄▁▁▃▃ ▃
  36.8 ms         Histogram: frequency by time          46 ms <

 Memory estimate: 9.70 MiB, allocs estimate: 157226.

In [11]:
@benchmark slowcontract(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 130 samples with 1 evaluation per sample.
 Range (min … max):  36.847 ms … 45.980 ms  ┊ GC (min … max): 0.00% … 15.39%
 Time  (median):     37.390 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   38.717 ms ±  2.938 ms  ┊ GC (mean ± σ):  3.25% ±  6.20%

   ▂▅█                                                         
  █████▇▄▃▄▃▄▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▄▄▃▆▃ ▃
  36.8 ms         Histogram: frequency by time        45.5 ms <

 Memory estimate: 9.70 MiB, allocs estimate: 157229.

In [12]:
norm(slowcontract(le = t_le, re=t_re, x = t_ac, O = t_o)- factory(le = t_le, re=t_re, x = t_ac, O = t_o))

0.0

## Multithreaded benchmark

gc becomes a severe bottleneck with many julia threads

In [13]:
virtspace = Rep[SU₂](0=>50, 1=> 30, 2=> 20, 3=>10, 4=> 5);
ospace = Rep[SU₂](0=>2, 1 => 2);
pspace = Rep[SU₂](1 => 1);


In [14]:
numblocks = 100;
const leftenvs = [rand(ComplexF64,virtspace*ospace',virtspace) for i in 1:numblocks];
const rightenvs = [rand(ComplexF64,virtspace*ospace,virtspace) for i in 1:numblocks];
const mpos = [rand(ComplexF64,ospace*pspace,pspace*ospace) for i in 1:numblocks];

ac = rand(ComplexF64,virtspace*pspace,virtspace);

In [15]:
using FLoops

t_le = leftenvs[1]
t_re = rightenvs[1]
t_o = mpos[1]

const fast_factory = ac_eff_malloc(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(ac),space(ac)))

function fastfun(x)
    @floop for (l,m,r) in zip(leftenvs,mpos,rightenvs)
        y = fast_factory(le = l,re = r, O = m, x = x)
        @reduce s += y
    end
    s
end

function slowfun(x)
    @floop for (le,O,re) in zip(leftenvs,mpos,rightenvs)
        @tensor allocator=malloc y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]
        @reduce s += y
    end
    s
end

slowfun (generic function with 1 method)

In [16]:
@benchmark fastfun(ac)

BenchmarkTools.Trial: 148 samples with 1 evaluation per sample.
 Range (min … max):  25.295 ms … 155.460 ms  ┊ GC (min … max):  0.00% … 83.22%
 Time  (median):     31.779 ms               ┊ GC (median):     7.74%
 Time  (mean ± σ):   33.889 ms ±  15.415 ms  ┊ GC (mean ± σ):  15.47% ± 12.82%

  ▁▄▇█▄                                                         
  █████▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂ ▂
  25.3 ms         Histogram: frequency by time          155 ms <

 Memory estimate: 29.68 MiB, allocs estimate: 187473.

In [17]:
@benchmark slowfun(ac)

BenchmarkTools.Trial: 147 samples with 1 evaluation per sample.
 Range (min … max):  25.526 ms … 155.922 ms  ┊ GC (min … max):  0.00% … 81.84%
 Time  (median):     32.945 ms               ┊ GC (median):     8.75%
 Time  (mean ± σ):   34.770 ms ±  15.257 ms  ┊ GC (mean ± σ):  16.86% ± 12.20%

  ▁▂▂▇█                                                         
  █████▇▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂ ▂
  25.5 ms         Histogram: frequency by time          152 ms <

 Memory estimate: 29.63 MiB, allocs estimate: 187773.